In [1]:
import json
import random
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ============================================================
# 設定
# ============================================================

TRAIN_FILE = Path("../data/train_strict_split.jsonl")
TEST_FILE = Path("../data/test_strict_split.jsonl")

OUTPUT_DIR = Path("char_bilstm_detector")

MODEL_FILE = OUTPUT_DIR / "model.pt"
VOCAB_FILE = OUTPUT_DIR / "char_vocab.json"
CONFIG_FILE = OUTPUT_DIR / "config.json"

# ------------------------------------------------------------
# 模型大小
# ------------------------------------------------------------

CHAR_EMBED_DIM = 16
CHAR_HIDDEN_DIM = 32
SENT_HIDDEN_DIM = 32

NUM_LABELS = 2

# ------------------------------------------------------------
# 訓練
# ------------------------------------------------------------

BATCH_SIZE = 64
EPOCHS = 30

LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-5

# 最多幾個 epoch 沒有改善就停止
PATIENCE = 5

# Token 最多取多少字元
MAX_CHAR_LENGTH = 64

# ------------------------------------------------------------
# Random seed
# ------------------------------------------------------------

SEED = 42


# ============================================================
# Random Seed
# ============================================================

def set_seed(seed=42):

    random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)


# ============================================================
# Device
# ============================================================

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ============================================================
# 特殊 Token
# ============================================================

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

PAD_ID = 0
UNK_ID = 1


# ============================================================
# 讀取 JSONL
# ============================================================

def load_jsonl(path):

    dataset = []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line_number, line in enumerate(f, 1):

            line = line.strip()

            if not line:
                continue

            try:
                item = json.loads(line)

            except json.JSONDecodeError as e:

                print(
                    f"JSON 錯誤："
                    f"{path} "
                    f"第 {line_number} 行"
                )

                raise e

            tokens = item["tokens"]
            labels = item["labels"]

            if len(tokens) != len(labels):

                raise ValueError(
                    f"{path} 第 {line_number} 行："
                    f"tokens 與 labels 長度不同"
                )

            dataset.append({
                "tokens": tokens,
                "labels": labels
            })

    return dataset


# ============================================================
# 建立 Character Vocabulary
# ============================================================

def build_char_vocab(dataset):

    chars = set()

    for sample in dataset:

        for token in sample["tokens"]:

            for char in token[:MAX_CHAR_LENGTH]:

                chars.add(char)

    # 排序讓 Vocabulary 固定
    chars = sorted(chars)

    vocab = {
        PAD_TOKEN: PAD_ID,
        UNK_TOKEN: UNK_ID
    }

    for char in chars:

        if char not in vocab:

            vocab[char] = len(vocab)

    return vocab


# ============================================================
# Token → Character IDs
# ============================================================

def token_to_ids(
    token,
    vocab
):

    chars = list(
        token[:MAX_CHAR_LENGTH]
    )

    if not chars:

        return [UNK_ID]

    ids = []

    for char in chars:

        ids.append(
            vocab.get(
                char,
                UNK_ID
            )
        )

    return ids


# ============================================================
# Dataset
# ============================================================

class SensitiveDataset(Dataset):

    def __init__(
        self,
        data,
        char_vocab
    ):

        self.data = data
        self.char_vocab = char_vocab

    def __len__(self):

        return len(self.data)

    def __getitem__(self, index):

        sample = self.data[index]

        tokens = sample["tokens"]
        labels = sample["labels"]

        char_ids = []

        for token in tokens:

            char_ids.append(
                token_to_ids(
                    token,
                    self.char_vocab
                )
            )

        labels = torch.tensor(
            labels,
            dtype=torch.long
        )

        return {
            "tokens": tokens,
            "char_ids": char_ids,
            "labels": labels
        }


# ============================================================
# Batch Collate
# ============================================================

def collate_fn(batch):

    batch_size = len(batch)

    # --------------------------------------------------------
    # 找最長 Sentence
    # --------------------------------------------------------

    max_sentence_length = max(
        len(item["char_ids"])
        for item in batch
    )

    # --------------------------------------------------------
    # 找最長 Token
    # --------------------------------------------------------

    max_char_length = 1

    for item in batch:

        for token_chars in item["char_ids"]:

            max_char_length = max(
                max_char_length,
                len(token_chars)
            )

    max_char_length = min(
        max_char_length,
        MAX_CHAR_LENGTH
    )

    # --------------------------------------------------------
    # Character Tensor
    #
    # [Batch, Sentence, Character]
    # --------------------------------------------------------

    char_ids = torch.full(
        (
            batch_size,
            max_sentence_length,
            max_char_length
        ),
        PAD_ID,
        dtype=torch.long
    )

    # --------------------------------------------------------
    # Character Length
    #
    # [Batch, Sentence]
    # --------------------------------------------------------

    char_lengths = torch.zeros(
        (
            batch_size,
            max_sentence_length
        ),
        dtype=torch.long
    )

    # --------------------------------------------------------
    # Token Mask
    #
    # [Batch, Sentence]
    # --------------------------------------------------------

    token_mask = torch.zeros(
        (
            batch_size,
            max_sentence_length
        ),
        dtype=torch.bool
    )

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    labels = torch.zeros(
        (
            batch_size,
            max_sentence_length
        ),
        dtype=torch.long
    )

    # --------------------------------------------------------
    # 填入
    # --------------------------------------------------------

    for batch_index, item in enumerate(batch):

        sentence_length = len(
            item["char_ids"]
        )

        token_mask[
            batch_index,
            :sentence_length
        ] = True

        labels[
            batch_index,
            :sentence_length
        ] = item["labels"]

        for token_index, token_chars in enumerate(
            item["char_ids"]
        ):

            token_chars = token_chars[
                :max_char_length
            ]

            length = len(token_chars)

            char_ids[
                batch_index,
                token_index,
                :length
            ] = torch.tensor(
                token_chars,
                dtype=torch.long
            )

            char_lengths[
                batch_index,
                token_index
            ] = length

    return {
        "char_ids": char_ids,
        "char_lengths": char_lengths,
        "token_mask": token_mask,
        "labels": labels
    }


# ============================================================
# Char BiLSTM + Sentence BiLSTM
# ============================================================

class CharBiLSTMTokenClassifier(
    nn.Module
):

    def __init__(
        self,
        char_vocab_size
    ):

        super().__init__()

        # ====================================================
        # Character Embedding
        # ====================================================

        self.char_embedding = nn.Embedding(
            num_embeddings=char_vocab_size,
            embedding_dim=CHAR_EMBED_DIM,
            padding_idx=PAD_ID
        )

        # ====================================================
        # Char BiLSTM
        # ====================================================

        self.char_lstm = nn.LSTM(
            input_size=CHAR_EMBED_DIM,
            hidden_size=CHAR_HIDDEN_DIM,
            batch_first=True,
            bidirectional=True
        )

        # ====================================================
        # Sentence BiLSTM
        # ====================================================

        self.sent_lstm = nn.LSTM(
            input_size=CHAR_HIDDEN_DIM * 2,
            hidden_size=SENT_HIDDEN_DIM,
            batch_first=True,
            bidirectional=True
        )

        # ====================================================
        # Classifier
        # ====================================================

        self.classifier = nn.Linear(
            SENT_HIDDEN_DIM * 2,
            NUM_LABELS
        )

    def forward(
        self,
        char_ids,
        char_lengths,
        token_mask
    ):

        batch_size = char_ids.size(0)

        sentence_length = char_ids.size(1)

        char_length = char_ids.size(2)

        # ====================================================
        # [B, S, C]
        #
        # ↓
        #
        # [B*S, C]
        # ====================================================

        flat_char_ids = char_ids.reshape(
            batch_size * sentence_length,
            char_length
        )

        flat_char_lengths = char_lengths.reshape(
            batch_size * sentence_length
        )

        # ====================================================
        # Embedding
        # ====================================================

        embedded = self.char_embedding(
            flat_char_ids
        )

        # ====================================================
        # 有些 Padding Token 長度是 0
        #
        # pack_padded_sequence 不接受 0
        # ====================================================

        safe_lengths = flat_char_lengths.clamp(
            min=1
        )

        # ----------------------------------------------------
        # Char BiLSTM
        # ----------------------------------------------------

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            safe_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.char_lstm(
            packed
        )

        # hidden:
        #
        # [2, B*S, Hidden]
        #
        # forward
        # backward

        forward_hidden = hidden[-2]

        backward_hidden = hidden[-1]

        token_vectors = torch.cat(
            [
                forward_hidden,
                backward_hidden
            ],
            dim=1
        )

        # ====================================================
        # [B*S, Hidden*2]
        #
        # ↓
        #
        # [B, S, Hidden*2]
        # ====================================================

        token_vectors = token_vectors.reshape(
            batch_size,
            sentence_length,
            CHAR_HIDDEN_DIM * 2
        )

        # ====================================================
        # Padding Token 清零
        # ====================================================

        token_vectors = token_vectors.masked_fill(
            ~token_mask.unsqueeze(-1),
            0
        )

        # ====================================================
        # Sentence BiLSTM
        # ====================================================

        sentence_lengths = token_mask.sum(
            dim=1
        ).clamp(
            min=1
        )

        packed_sentence = nn.utils.rnn.pack_padded_sequence(
            token_vectors,
            sentence_lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.sent_lstm(
            packed_sentence
        )

        sentence_output, _ = nn.utils.rnn.pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=sentence_length
        )

        # ====================================================
        # Classifier
        # ====================================================

        logits = self.classifier(
            sentence_output
        )

        return logits


# ============================================================
# 計算 Class Weight
# ============================================================

def calculate_class_weights(dataset):

    count_0 = 0
    count_1 = 0

    for sample in dataset:

        for label in sample["labels"]:

            if label == 0:
                count_0 += 1

            elif label == 1:
                count_1 += 1

    total = count_0 + count_1

    # 平衡權重
    weight_0 = total / (2 * count_0)

    weight_1 = total / (2 * count_1)

    weights = torch.tensor(
        [
            weight_0,
            weight_1
        ],
        dtype=torch.float32
    )

    return (
        weights,
        count_0,
        count_1
    )


# ============================================================
# Train 一個 Epoch
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion
):

    model.train()

    total_loss = 0.0

    correct = 0
    total = 0

    for batch in loader:

        char_ids = batch[
            "char_ids"
        ].to(DEVICE)

        char_lengths = batch[
            "char_lengths"
        ].to(DEVICE)

        token_mask = batch[
            "token_mask"
        ].to(DEVICE)

        labels = batch[
            "labels"
        ].to(DEVICE)

        # ----------------------------------------------------
        # Gradient
        # ----------------------------------------------------

        optimizer.zero_grad()

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        logits = model(
            char_ids,
            char_lengths,
            token_mask
        )

        # ----------------------------------------------------
        # Loss
        # ----------------------------------------------------

        active_logits = logits[
            token_mask
        ]

        active_labels = labels[
            token_mask
        ]

        loss = criterion(
            active_logits,
            active_labels
        )

        # ----------------------------------------------------
        # Backward
        # ----------------------------------------------------

        loss.backward()

        # 防止梯度爆炸
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        # ----------------------------------------------------
        # Statistics
        # ----------------------------------------------------

        total_loss += loss.item()

        predictions = active_logits.argmax(
            dim=-1
        )

        correct += (
            predictions == active_labels
        ).sum().item()

        total += active_labels.numel()

    avg_loss = total_loss / len(loader)

    accuracy = (
        correct / total
        if total > 0
        else 0
    )

    return avg_loss, accuracy


# ============================================================
# Evaluate
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader,
    criterion
):

    model.eval()

    total_loss = 0.0

    tp = 0
    tn = 0
    fp = 0
    fn = 0

    for batch in loader:

        char_ids = batch[
            "char_ids"
        ].to(DEVICE)

        char_lengths = batch[
            "char_lengths"
        ].to(DEVICE)

        token_mask = batch[
            "token_mask"
        ].to(DEVICE)

        labels = batch[
            "labels"
        ].to(DEVICE)

        # ----------------------------------------------------
        # Forward
        # ----------------------------------------------------

        logits = model(
            char_ids,
            char_lengths,
            token_mask
        )

        active_logits = logits[
            token_mask
        ]

        active_labels = labels[
            token_mask
        ]

        loss = criterion(
            active_logits,
            active_labels
        )

        total_loss += loss.item()

        predictions = active_logits.argmax(
            dim=-1
        )

        # ----------------------------------------------------
        # Confusion Matrix
        # ----------------------------------------------------

        tp += (
            (predictions == 1)
            & (active_labels == 1)
        ).sum().item()

        tn += (
            (predictions == 0)
            & (active_labels == 0)
        ).sum().item()

        fp += (
            (predictions == 1)
            & (active_labels == 0)
        ).sum().item()

        fn += (
            (predictions == 0)
            & (active_labels == 1)
        ).sum().item()

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    total = tp + tn + fp + fn

    accuracy = (
        (tp + tn) / total
        if total > 0
        else 0
    )

    precision = (
        tp / (tp + fp)
        if tp + fp > 0
        else 0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn > 0
        else 0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall > 0
        else 0
    )

    avg_loss = total_loss / len(loader)

    metrics = {
        "loss": avg_loss,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

    return metrics


# ============================================================
# 儲存模型
# ============================================================

def save_model(
    model,
    char_vocab
):

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    # --------------------------------------------------------
    # Model
    # --------------------------------------------------------

    torch.save(
        model.state_dict(),
        MODEL_FILE
    )

    # --------------------------------------------------------
    # Vocabulary
    # --------------------------------------------------------

    with open(
        VOCAB_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            char_vocab,
            f,
            ensure_ascii=False,
            indent=2
        )

    # --------------------------------------------------------
    # Config
    # --------------------------------------------------------

    config = {
        "char_embed_dim": CHAR_EMBED_DIM,
        "char_hidden_dim": CHAR_HIDDEN_DIM,
        "sent_hidden_dim": SENT_HIDDEN_DIM,
        "num_labels": NUM_LABELS,
        "max_char_length": MAX_CHAR_LENGTH,
        "pad_id": PAD_ID,
        "unk_id": UNK_ID
    }

    with open(
        CONFIG_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            config,
            f,
            indent=2
        )


# ============================================================
# 主程式
# ============================================================

def main():

    print()
    print("=" * 60)
    print("正在讀取訓練資料...")
    print("=" * 60)

    # ========================================================
    # 檢查檔案
    # ========================================================

    if not TRAIN_FILE.exists():

        raise FileNotFoundError(
            f"找不到：{TRAIN_FILE}"
        )

    if not TEST_FILE.exists():

        raise FileNotFoundError(
            f"找不到：{TEST_FILE}"
        )

    # ========================================================
    # Load Dataset
    # ========================================================

    train_data = load_jsonl(
        TRAIN_FILE
    )

    test_data = load_jsonl(
        TEST_FILE
    )

    print(
        f"Train 句子數量：{len(train_data)}"
    )

    print(
        f"Test 句子數量：{len(test_data)}"
    )

    # ========================================================
    # Statistics
    # ========================================================

    train_token_count = sum(
        len(x["tokens"])
        for x in train_data
    )

    train_label_1 = sum(
        sum(x["labels"])
        for x in train_data
    )

    train_label_0 = (
        train_token_count
        - train_label_1
    )

    print(
        f"Train Token 總數："
        f"{train_token_count}"
    )

    print(
        f"Train Label 1："
        f"{train_label_1}"
    )

    print(
        f"Train Label 0："
        f"{train_label_0}"
    )

    # ========================================================
    # Character Vocabulary
    #
    # 注意：
    #
    # Vocabulary 只從 TRAIN 建立
    #
    # Test 裡沒見過的字元
    # → UNK
    #
    # 這才是真正的 strict test
    # ========================================================

    char_vocab = build_char_vocab(
        train_data
    )

    print(
        f"Character Vocabulary 大小："
        f"{len(char_vocab)}"
    )

    # ========================================================
    # Dataset
    # ========================================================

    train_dataset = SensitiveDataset(
        train_data,
        char_vocab
    )

    test_dataset = SensitiveDataset(
        test_data,
        char_vocab
    )

    # ========================================================
    # DataLoader
    # ========================================================

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn
    )

    # ========================================================
    # Model
    # ========================================================

    model = CharBiLSTMTokenClassifier(
        char_vocab_size=len(char_vocab)
    )

    model.to(DEVICE)

    # ========================================================
    # Parameters
    # ========================================================

    parameter_count = sum(
        p.numel()
        for p in model.parameters()
    )

    print(
        f"模型參數數量："
        f"{parameter_count:,}"
    )

    print(
        f"使用裝置："
        f"{DEVICE}"
    )

    # ========================================================
    # Class Weight
    # ========================================================

    class_weights, count_0, count_1 = (
        calculate_class_weights(
            train_data
        )
    )

    class_weights = class_weights.to(
        DEVICE
    )

    print()
    print(
        f"Class 0：{count_0}"
    )

    print(
        f"Class 1：{count_1}"
    )

    print(
        f"Class Weight："
        f"{class_weights.tolist()}"
    )

    # ========================================================
    # Loss
    # ========================================================

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    # ========================================================
    # Optimizer
    # ========================================================

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    # ========================================================
    # Training
    # ========================================================

    print()
    print("=" * 60)
    print("開始訓練...")
    print("=" * 60)

    best_f1 = -1.0

    patience_counter = 0

    for epoch in range(
        1,
        EPOCHS + 1
    ):

        # ====================================================
        # Train
        # ====================================================

        train_loss, train_accuracy = (
            train_one_epoch(
                model,
                train_loader,
                optimizer,
                criterion
            )
        )

        # ====================================================
        # Test
        # ====================================================

        test_metrics = evaluate(
            model,
            test_loader,
            criterion
        )

        # ====================================================
        # Print
        # ====================================================

        print()

        print(
            f"Epoch "
            f"{epoch:03d}/{EPOCHS}"
        )

        print(
            f"  Train Loss      : "
            f"{train_loss:.6f}"
        )

        print(
            f"  Train Accuracy  : "
            f"{train_accuracy:.2%}"
        )

        print(
            f"  Test Loss       : "
            f"{test_metrics['loss']:.6f}"
        )

        print(
            f"  Test Accuracy   : "
            f"{test_metrics['accuracy']:.2%}"
        )

        print(
            f"  Precision       : "
            f"{test_metrics['precision']:.2%}"
        )

        print(
            f"  Recall          : "
            f"{test_metrics['recall']:.2%}"
        )

        print(
            f"  F1              : "
            f"{test_metrics['f1']:.2%}"
        )

        print(
            f"  TP={test_metrics['tp']} "
            f"TN={test_metrics['tn']} "
            f"FP={test_metrics['fp']} "
            f"FN={test_metrics['fn']}"
        )

        # ====================================================
        # Best Model
        #
        # 使用 F1，而不是 Accuracy
        # ====================================================

        if test_metrics["f1"] > best_f1:

            best_f1 = test_metrics["f1"]

            patience_counter = 0

            save_model(
                model,
                char_vocab
            )

            print(
                f"  ★ 儲存最佳模型 "
                f"(F1={best_f1:.2%})"
            )

        else:

            patience_counter += 1

            print(
                f"  沒有改善 "
                f"({patience_counter}/{PATIENCE})"
            )

        # ====================================================
        # Early Stopping
        # ====================================================

        if patience_counter >= PATIENCE:

            print()
            print(
                "Early Stopping："
                "Test F1 已經沒有改善。"
            )

            break

    # ========================================================
    # Finish
    # ========================================================

    print()
    print("=" * 60)
    print("訓練完成")
    print("=" * 60)

    print(
        f"最佳 Test F1："
        f"{best_f1:.2%}"
    )

    print()
    print("模型已儲存：")

    print(
        f"  {MODEL_FILE}"
    )

    print(
        f"  {VOCAB_FILE}"
    )

    print(
        f"  {CONFIG_FILE}"
    )

    print()
    print(
        "接下來可以執行："
    )

    print(
        "  python predict.py"
    )


# ============================================================
# Entry
# ============================================================

if __name__ == "__main__":

    main()


正在讀取訓練資料...
Train 句子數量：5000
Test 句子數量：1000
Train Token 總數：29473
Train Label 1：3066
Train Label 0：26407
Character Vocabulary 大小：72
模型參數數量：39,170
使用裝置：cpu

Class 0：26407
Class 1：3066
Class Weight：[0.5580527782440186, 4.806425094604492]

開始訓練...

Epoch 001/30
  Train Loss      : 0.378201
  Train Accuracy  : 81.58%
  Test Loss       : 0.055001
  Test Accuracy   : 97.04%
  Precision       : 75.87%
  Recall          : 99.84%
  F1              : 86.22%
  TP=610 TN=5789 FP=194 FN=1
  ★ 儲存最佳模型 (F1=86.22%)

Epoch 002/30
  Train Loss      : 0.037002
  Train Accuracy  : 98.52%
  Test Loss       : 0.043321
  Test Accuracy   : 97.27%
  Precision       : 77.31%
  Recall          : 99.84%
  F1              : 87.14%
  TP=610 TN=5804 FP=179 FN=1
  ★ 儲存最佳模型 (F1=87.14%)

Epoch 003/30
  Train Loss      : 0.018279
  Train Accuracy  : 99.15%
  Test Loss       : 0.063844
  Test Accuracy   : 97.54%
  Precision       : 79.50%
  Recall          : 99.02%
  F1              : 88.19%
  TP=605 TN=5827 FP=156 FN=6
  